In [1]:
!pip install timm albumentations flask

In [33]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
from glob import glob
import json

model_dir = '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/train_results'
annotations_path = os.path.join(model_dir, 'annotations.json')
with open(annotations_path) as f:
    annotations = json.load(f)

print(f"이미지 개수: {len(annotations)}")

이미지 개수: 37801


In [3]:
annotations[:2]

[{'image_path': '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/리조트/995453.jpg',
  'image_id': 1,
  'box': [297, 417, 481, 706],
  'detail_category': '티셔츠',
  'color': '핑크',
  'fit': '노멀',
  'length': '노멀'},
 {'image_path': '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/페미닌/868254.jpg',
  'image_id': 0,
  'box': [319, 552, 499, 873],
  'detail_category': '드레스',
  'color': '민트',
  'fit': '노멀',
  'length': '미니'}]

In [6]:
!FLASK_ENV=development FLASK_APP='/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py' flask run

model.safetensors: 100% 21.4M/21.4M [00:03<00:00, 6.40MB/s]
/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py:62: UserWarning: Argument(s) 'always_apply' are not valid for transform MaxSizeTransform
  A.LongestMaxSize(max_size=224,
/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py:64: UserWarning: Argument(s) 'always_apply' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=224,
 * Serving Flask app '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
Exception ignored in atexit callback: <bound method TMonitor.exit of <TMonitor(Thread-1, stopped daemon 138795693372992)>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/_monitor.py", line 44, in exit
 

In [7]:
import subprocess

flask_command = ["python3", "/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py"]

flask_process = subprocess.Popen(flask_command)

In [8]:
import requests
import io
from PIL import Image
import random
from tqdm import tqdm
import numpy as np

feature_map = {}
margin = 0.1
random.shuffle(annotations)

for annot in tqdm(annotations[:10]):
    image_path = annot['image_path']
    image_id = annot['image_id']
    image = Image.open(image_path).convert('RGB')
    box = annot['box']

    w = box[2] - box[0]
    h = box[3] - box[1]

    new_box = [
        int(box[0] - w * margin),
        int(box[1] - h * margin),
        int(box[2] + w * margin),
        int(box[3] + h * margin),
    ]
    cropped_image = image.crop(new_box)

    image_bytes = io.BytesIO()
    cropped_image.save(image_bytes, format='JPEG')

    files = {'file': ('test_image.jpg', image_bytes.getvalue())}

    resp = requests.post(url="http://localhost:5000/predict", files=files)

    result = resp.json()
    feature_map[f"{image_path}_{image_id}"] = result["feature"]


100%|██████████| 10/10 [01:33<00:00,  9.38s/it]


In [9]:
flask_process.terminate()

In [10]:
feature_map.keys()

dict_keys(['/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/페미닌/1230878.jpg_0', '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/모던/1336525.jpg_1', '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/컨트리/1054643.jpg_1', '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/스트리트/1219179.jpg_0', '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/스트리트/613555.jpg_0', '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/스트리트/339502.jpg_1', '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/리조트/1132245.jpg_0', '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/소피스트케이티ᄃ

In [12]:
import numpy as np

feat = np.array(feature_map['/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/스트리트/1219179.jpg_0'])
feat.shape

(1, 1280)

In [13]:
print(feat)

[[ 0.41702044 -0.08474134 -0.13909324 ... -0.05674152  0.03261815
   0.02162692]]


In [24]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

class BatchJsonDataset(Dataset):
    def __init__(self, annotations, margin=0.1, transform=None):
        self.annotations = annotations
        self.margin = margin
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        annot = self.annotations[index]
        image_path = os.path.join(annot['image_path'])
        image_id = annot['image_id']
        image_path_id = f"{image_path}_{image_id}"
        image = Image.open(image_path).convert("RGB")
        box = annot['box']

        w = box[2] - box[0]
        h = box[3] - box[1]

        new_box = [int(box[0] - w * self.margin),
                   int(box[1] - h * self.margin),
                   int(box[2] + w * self.margin),
                   int(box[3] + h * self.margin)]

        cropped_image = image.crop(new_box)

        if self.transform:
            try:
                cropped_image = self.transform(image=np.array(cropped_image))['image']
                image = self.transform(image=np.array(image))['image']
            except:
                black_image = Image.new('RGB', (224, 224), (0, 0, 0))
                cropped_image = self.transfrom(image=np.array(black_image))['image']
                print(idx, annot)


        return cropped_image, image, image_path_id

In [35]:
import torch.nn as nn
import timm

class BranchClassifier(nn.Module):
    def __init__(self, num_classes_list, num_branches=4, model_name='efficientnet_b0', pretrained=True):
        super(BranchClassifier, self).__init__()

        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1280, 256),
                nn.SiLU(),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes_list[i])
            )
            for i in range(num_branches)
        ])

    def forward(self, x):
        features = self.backbone(x)
        outputs = [branch(features) for branch in self.branches]

        return outputs, features

In [36]:
import os
import torch
import json
import albumentations as A
from albumentations.pytorch import ToTensorV2

model_dir = '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/train_results'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_branches = 4

with open(os.path.join(model_dir, 'detail_category_list.json'), 'r') as json_f:
    detail_category_list = json.load(json_f)
with open(os.path.join(model_dir, 'color_list.json'), 'r') as json_f:
    color_list = json.load(json_f)
with open(os.path.join(model_dir, 'fit_list.json'), 'r') as json_f:
    fit_list = json.load(json_f)
with open(os.path.join(model_dir, 'length_list.json'), 'r') as json_f:
    length_list = json.load(json_f)

num_classes_list = [len(detail_category_list), len(color_list), len(fit_list), len(length_list)]
model = BranchClassifier(num_classes_list = num_classes_list).eval().to(device)
weight = torch.load(os.path.join(model_dir, 'best_model.pth'), map_location='cpu')
model.load_state_dict(weight)

transform = A.Compose([
    A.LongestMaxSize(max_size=224, always_apply=True),
    A.PadIfNeeded(min_height=224, min_width=224, always_apply=True, border_mode=0),
    A.Normalize(p=1.0, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


In [39]:
dataset = BatchJsonDataset(annotations, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=128, num_workers=4, shuffle=True, prefetch_factor=2, pin_memory=True)

In [40]:
feature_map = {}
model.eval()

with torch.no_grad():
    for idx, (cropped_images, images, image_path_id) in tqdm(enumerate(dataloader)):
        cropped_images = cropped_images.to(device)
        outputs, features = model(cropped_images)
        for batch_idx in range(len(features)):
            feature_map[image_path_id[batch_idx]] = features[batch_idx].cpu().numpy().tolist()

34it [12:47, 22.58s/it]
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dbf9c4542c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1671, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._sel

KeyboardInterrupt: 

In [43]:
len(feature_map)

4352

In [41]:
list(feature_map.keys())[:2]

['/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/스트리트/209374.jpg_1',
 '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/Datasets/part3_chapter03/images/스트리트/138691.jpg_0']

In [44]:
vector_dir = '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/vectors'
os.makedirs (vector_dir, exist_ok=True)
with open('/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/vectors/feature_map.json', 'w') as f:
    json.dump(feature_map, f, indent='\t', ensure_ascii=False)

In [45]:
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA
import pickle

image_path_id_list = []
feature_list = []

for key, value in feature_map.items():
    image_path_id_list.append(key)
    feature_list.append(value)

feature_list = np.array(feature_list)

normalizer = Normalizer(norm='l2')
norm_feature_list = normalizer.fit_transform(feature_list)

pca = PCA(n_components=128)
comp_feature_list = pca.fit_transform(norm_feature_list)

vector_dir = '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/vectors'
pca_save_path = os.path.join(vector_dir, 'pca_model.pkl')
with open(pca_save_path, 'wb') as f:
    pickle.dump(pca, f)

In [46]:
feature_list[0]

array([-0.08624484, -0.08798609, -0.06392018, ...,  1.64621878,
       -0.1127868 , -0.06170754])

In [47]:
norm_feature_list[0]

array([-0.00452757, -0.00461898, -0.0033556 , ...,  0.08642113,
       -0.00592094, -0.00323945])

In [48]:
norm_feature_list[0].shape

(1280,)

In [50]:
comp_feature_list[0].shape

(128,)

In [51]:
imagenet_model = timm.create_model('efficientnet_b0', pretrained=True, num_class=0)
imagenet_model.eval()
imagenet_model = imagenet_model.to(device)

sample_input = torch.randn(1, 3, 224, 224).to(device)
imagenet_model(sample_input).shape

TypeError: EfficientNet.__init__() got an unexpected keyword argument 'num_class'